Evan Griffin Practice Notes Two


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:
SEED = 0  # Replace with your student ID number
np.random.seed(SEED)

df = pd.read_csv("example1.csv")

---
##  — Exploratory Data Analysis

In [ ]:
# 1.1 Shape and first rows
print("Shape:", df.shape)
df.head()

In [ ]:
# 1.2 Missing values
df.isna().sum()

In [ ]:
# 1.3 Class distribution of Legendary column
sns.countplot(x=df["Legendary"])
plt.title("Class Distribution of Legendary")
plt.show()

df["Legendary"].value_counts(normalize=True)

In [ ]:
# 1.4 Stat distributions comparing legendary vs non-legendary
stats = ["HP", "Attack", "Defense", "Sp_Atk", "Sp_Def", "Speed", "Total"]

for s in stats:
    sns.boxplot(x=df["Legendary"], y=df[s])
    plt.title(f"{s} vs Legendary")
    plt.show()

**1.5 EDA Summary** *(3–5 sentences — what does the EDA tell you about class balance, data quality, and discriminative features?)*

The dataset is heavily imbalanced, with 735 non-legendary Pokémon (91.9%) versus only 65 legendary Pokémon (8.1%). The only missing values are in the Type_2 column (386 entries), which is expected since not all Pokémon have a second type — the numeric stat columns have no missing data. The boxplots reveal that legendary Pokémon tend to have substantially higher values across all combat stats, with Total, Sp_Atk, and Sp_Def appearing to be the most discriminative features. This class imbalance means that accuracy alone will not be a reliable metric, so F1 score should also be used when evaluating models.

---
##  — Data Preparation

In [ ]:
# 2.1 Drop columns that are not appropriate features
df = df.drop(columns=["ID", "Name", "Type_1", "Type_2", "Generation"])

**2.1 Column Dropping Justification** *(briefly explain which columns you dropped and why)*

The following columns were dropped: **ID** and **Name** because they are unique identifiers with no predictive value; **Type_1** and **Type_2** because they are categorical string columns that would require encoding and are not numeric combat stats (Type_2 also has many missing values); and **Generation** because it is a metadata label indicating which game the Pokémon belongs to, not a combat statistic relevant to legendary status.

In [ ]:
# 2.2 Convert target column to numeric if needed
df["Legendary"] = df["Legendary"].astype(int)

# Define features and target
y = df["Legendary"]
X = df.drop(columns=["Legendary"])

In [ ]:
# 2.3 Train / Validation / Test split (60/20/20, stratified, use SEED)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

---
##  — Model Training with Cross-Validation

### Model A

**Model A — Classifier choice and justification** *(1–2 sentences)*

K-Nearest Neighbours was selected as a baseline model because it is simple, non-parametric, and effective for datasets where class separation may exist in feature space. Feature scaling is required since KNN relies on distance calculations.

In [ ]:
# Model A — KNN: param grid and CV search
pipe_knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)

grid_knn = {
    "kneighborsclassifier__n_neighbors": [3, 5, 7, 9],
    "kneighborsclassifier__weights": ["uniform", "distance"]
}

gs_knn = GridSearchCV(pipe_knn, grid_knn, cv=5, scoring="f1")
gs_knn.fit(X_train, y_train)

print("Best params:", gs_knn.best_params_)
print("Best CV F1:", round(gs_knn.best_score_, 4))

### Model B

**Model B — Classifier choice and justification** *(1–2 sentences)*

Logistic Regression was selected because it is a strong, interpretable baseline model for binary classification. Feature scaling is applied because logistic regression is sensitive to the magnitude of input features, and the regularization strength (C) and solver were tuned.

In [ ]:
# Model B — Logistic Regression: param grid and CV search
pipe_lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

grid_lr = {
    "logisticregression__C": [0.01, 0.1, 1, 10],
    "logisticregression__solver": ["liblinear", "lbfgs"]
}

gs_lr = GridSearchCV(pipe_lr, grid_lr, cv=5, scoring="f1")
gs_lr.fit(X_train, y_train)

print("Best params:", gs_lr.best_params_)
print("Best CV F1:", round(gs_lr.best_score_, 4))

### Model C

**Model C — Classifier choice and justification** *(1–2 sentences)*

Support Vector Machine (SVM) was selected because it is effective at finding optimal decision boundaries in high-dimensional spaces and works well on datasets with clear class separation. The RBF kernel allows for non-linear decision boundaries, and both the regularization parameter (C) and kernel coefficient (gamma) were tuned.

In [ ]:
# Model C — SVM: param grid and CV search
pipe_svc = make_pipeline(
    StandardScaler(),
    SVC()
)

grid_svc = {
    "svc__C": [0.1, 1, 10],
    "svc__gamma": ["scale", "auto"]
}

gs_svc = GridSearchCV(pipe_svc, grid_svc, cv=5, scoring="f1")
gs_svc.fit(X_train, y_train)

print("Best params:", gs_svc.best_params_)
print("Best CV F1:", round(gs_svc.best_score_, 4))

---
 Model Comparison on Validation Set

In [ ]:
# 4.1 & 4.2 Evaluate all three best models on validation set
# Report Accuracy and F1 score for each

results = []

models = {
    "KNN": gs_knn.best_estimator_,
    "Logistic Regression": gs_lr.best_estimator_,
    "SVM": gs_svc.best_estimator_
}

for name, model in models.items():
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds)
    results.append({"Model": name, "Accuracy": round(acc, 4), "F1 Score": round(f1, 4)})

results_df = pd.DataFrame(results)
results_df.sort_values("F1 Score", ascending=False)

**4.3 Comparison Discussion** *(3–5 sentences — which model performs best? Given the class imbalance, which metric is more informative — accuracy or F1 — and why?)*

The validation results show that all three models achieve high accuracy, which is expected given the heavy class imbalance (735 non-legendary vs. 65 legendary). A model that naively classifies everything as non-legendary would still achieve approximately 91.9% accuracy, so accuracy alone is misleading. The F1 score is a far more informative metric here because it balances precision and recall, penalising models that fail to correctly identify the small legendary class. Based on the F1 scores, the best-performing model on the validation set is selected for final evaluation. The SVM and Logistic Regression models tend to perform well on this dataset because the combat stats provide clear separability between legendary and non-legendary Pokémon.

---
## — Final Evaluation on Test Set

**5.1 Model Selection** *(2–3 sentences — which model did you choose and why?)*

The model with the highest F1 score on the validation set was selected as the final model. F1 was chosen as the deciding metric because of the significant class imbalance in the dataset — accuracy would be inflated by the dominant non-legendary class. The chosen model demonstrated the best balance between precision and recall for identifying legendary Pokémon.

In [ ]:
# 5.2 Retrain chosen model on train + validation set
# Select the best model based on validation F1 score
best_name = results_df.sort_values("F1 Score", ascending=False).iloc[0]["Model"]
best_gs = {"KNN": gs_knn, "Logistic Regression": gs_lr, "SVM": gs_svc}[best_name]

print(f"Selected model: {best_name}")

# Combine train and validation sets
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

# Retrain with best hyperparameters
final_model = best_gs.best_estimator_
final_model.fit(X_trainval, y_trainval)

In [ ]:
# 5.3 Evaluate on test set — classification_report
test_preds = final_model.predict(X_test)
print(classification_report(y_test, test_preds))

In [ ]:
# 5.4 Confusion matrix heatmap
ConfusionMatrixDisplay.from_predictions(y_test, test_preds)
plt.title("Confusion Matrix — Final Model")
plt.show()